In [ ]:
import sys
from pathlib import Path
for _root in [Path.cwd(), *Path.cwd().parents]:
    if (_root / "paths.py").exists():
        sys.path.insert(0, str(_root))
        break
else:
    raise RuntimeError(
        "Could not find Merge project root (paths.py). Run Jupyter with cwd Merge, notebooks/, or attribution/."
    )
import paths


In [6]:
!pip install transformers accelerate
# !pip install torch
# !pip install huggingface_hub 
# !pip uninstall -y numpy transformers tokenizers torch

  Using cached transformers-4.57.1-py3-none-any.whl.metadata (43 kB)
  Using cached accelerate-1.10.1-py3-none-any.whl.metadata (19 kB)
  Using cached regex-2025.9.18-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl.metadata (40 kB)
  Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.8 kB)
  Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.1 kB)
Using cached transformers-4.57.1-py3-none-any.whl (12.0 MB)
Using cached tokenizers-0.22.1-cp39-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (3.3 MB)
Using cached accelerate-1.10.1-py3-none-any.whl (374 kB)
Using cached regex-2025.9.18-cp310-cp310-manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_28_x86_64.whl (789 kB)
Using cached safetensors-0.6.2-cp38-abi3-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (485 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [accelerate]5 [accelerate]s]


In [4]:
!pip uninstall ipywidgets tqdm -y
!pip install tqdm==4.66.4

Found existing installation: ipywidgets 8.1.7
Uninstalling ipywidgets-8.1.7:
  Successfully uninstalled ipywidgets-8.1.7
Found existing installation: tqdm 4.67.1
Uninstalling tqdm-4.67.1:
  Successfully uninstalled tqdm-4.67.1


In [1]:
from huggingface_hub import login
login(new_session=False)

In [7]:
import os

# Completely disable any interactive progress bars and widget attempts
os.environ["HF_HUB_DISABLE_PROGRESS_BARS"] = "1"
os.environ["HF_HUB_ENABLE_HF_TRANSFER"] = "0"
os.environ["TRANSFORMERS_NO_ADVISORY_WARNINGS"] = "1"
os.environ["DISABLE_TQDM"] = "1"

# Optional: suppress Hugging Face logging
from transformers.utils import logging
logging.set_verbosity_error()

# Only then import Hugging Face components
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="epfl-llm/meditron-70b",
    token=True,              # or token="hf_xxxxxxx"
    device_map="auto",
    torch_dtype="auto"
)

prompt = "Explain the mechanism of action of beta-blockers in simple terms."
output = pipe(prompt, max_new_tokens=150)
print(output[0]["generated_text"])


Fetching 8 files:   0%|          | 0/8 [00:00<?, ?it/s]

LookupError: <ContextVar name='shell_parent' at 0x146dcaa51c10>

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM

tokenizer = AutoTokenizer.from_pretrained("epfl-llm/meditron-70b", use_fast=True)
model = AutoModelForCausalLM.from_pretrained("epfl-llm/meditron-70b", device_map="auto", torch_dtype="auto")

tokenizer_config.json:   0%|          | 0.00/4.08k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

  2025-10-19T13:38:40.087394Z ERROR  Python exception updating progress:, error: PyErr { type: <class 'LookupError'>, value: LookupError(<ContextVar name='shell_parent' at 0x146dcaa51c10>), traceback: Some(<traceback object at 0x146c46ed0980>) }, caller: "src/progress_update.rs:313"
    at /home/runner/work/xet-core/xet-core/error_printer/src/lib.rs:28



In [ ]:
import transformers
import torch

model_id = "epfl-llm/meditron-70b"

pipeline = transformers.pipeline(
    "text-generation",
    model=model_id,
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="auto",
)

messages = [
    {"role": "system", "content": "You are an expert and experienced from the healthcare and biomedical domain with extensive medical knowledge and practical experience. Your name is OpenBioLLM, and you were developed by Saama AI Labs. who's willing to help answer the user's query with explanation. In your explanation, leverage your deep medical expertise such as relevant anatomical structures, physiological processes, diagnostic criteria, treatment guidelines, or other pertinent medical concepts. Use precise medical terminology while still aiming to make the explanation clear and accessible to a general audience."},
    {"role": "user", "content": "How can i split a 3mg or 4mg waefin pill so i can get a 2.5mg pill?"},
]

prompt = pipeline.tokenizer.apply_chat_template(
        messages, 
        tokenize=False, 
        add_generation_prompt=True
)

terminators = [
    pipeline.tokenizer.eos_token_id,
    pipeline.tokenizer.convert_tokens_to_ids("<|eot_id|>")
]

outputs = pipeline(
    prompt,
    max_new_tokens=256,
    eos_token_id=terminators,
    do_sample=True,
    temperature=0.0,
    top_p=0.9,
)
print(outputs[0]["generated_text"][len(prompt):])